# 2.7 Fancy Indexing (Gelişmiş İndeksleme)

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/02-numpy/07-fancy-indexing.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 07 Fancy Indexing

Önceki bölümlerde basit indeksler (arr[0]), dilimler (arr[:5]) ve Boolean maskeler (arr[arr > 0]) ile dizilerin bölümlerine erişmeyi gördük. Bu bölümde fancy indexing (veya vektörize indeksleme): tek skaler yerine indeks dizisi geçirerek birden çok elemana aynı anda erişim.

## Fancy Indexing'i Keşfetmek

Kavramsal olarak basit: birden fazla elemana erişmek için indeks dizisi geçirmek.


In [ ]:
# fancy_x.py
import numpy as np
rng = np.random.default_rng(seed=1701)

x = rng.integers(100, size=10)
print(x)



Üç farklı elemana erişmek:


In [ ]:
# fancy_manual.py
[x[3], x[7], x[2]]



Alternatif: tek bir liste veya indeks dizisi:


In [ ]:
# fancy_ind.py
ind = [3, 7, 4]
x[ind]



İndeks dizileri kullanıldığında sonucun şekli, indekslenen dizinin değil indeks dizilerinin şeklini yansıtır:


In [ ]:
# fancy_2d_ind.py
ind = np.array([[3, 7],
                [4, 5]])
x[ind]



Fancy indexing çok boyutlu dizilerde de çalışır:


In [ ]:
# fancy_X.py
X = np.arange(12).reshape((3, 4))
X



Standart indekslemede olduğu gibi ilk indeks satır, ikinci sütun:


In [ ]:
# fancy_row_col.py
row = np.array([0, 1, 2])
col = np.array([2, 1, 3])
X[row, col]



İlk değer X[0, 2], ikinci X[1, 1], üçüncü X[2, 3]. İndeks eşleştirmesi 2.5 Broadcasting kurallarına uyar. Sütun vektörü + satır vektörü birleştirilirse iki boyutlu sonuç:


In [ ]:
# fancy_broadcast_ind.py
X[row[:, np.newaxis], col]



Her satır değeri her sütun vektörüyle eşleşir — aritmetik broadcasting'deki gibi:


In [ ]:
# fancy_broadcast_demo.py
row[:, np.newaxis] * col



> **Not**
>

## Birleşik İndeksleme

Fancy indexing diğer indeksleme şemalarıyla birleştirilebilir:


In [ ]:
# print_X.py
print(X)



Fancy + basit indeks:


In [ ]:
# fancy_simple.py
X[2, [2, 0, 1]]



Fancy + dilimleme:


In [ ]:
# fancy_slice.py
X[1:, [2, 0, 1]]



Fancy + maskeleme:


In [ ]:
# fancy_mask.py
mask = np.array([True, False, True, False])
X[row[:, np.newaxis], mask]



Tüm bu seçenekler dizilere verimli erişim ve değiştirme için esnek bir araç seti sunar.

## Örnek: Rastgele Nokta Seçimi

Fancy indexing'in yaygın kullanımı: matristen satır alt kümesi seçmek. $N 	imes D$ boyutlu matris — $D$ boyutlu $N$ nokta. İki boyutlu normal dağılımdan noktalar:


In [ ]:
# random_points.py
mean = [0, 0]
cov = [[1, 2],
       [2, 5]]
X = rng.multivariate_normal(mean, cov, 100)
X.shape



Kitap bu noktaları scatter plot ile gösterir (Bölüm 4 — Matplotlib). 20 rastgele nokta seçmek için tekrarsız 20 indeks:


In [ ]:
# random_indices.py
indices = np.random.choice(X.shape[0], 20, replace=False)
indices



In [ ]:
# random_selection.py
selection = X[indices]  # fancy indexing
selection.shape



Seçilen noktalar büyük dairelerle üst üste çizilebilir. Bu strateji veri kümesini hızlıca bölmede kullanılır — örneğin istatistiksel modellerde eğitim/test ayrımı (kitap Bölüm 5 — Hyperparameters and Model Validation).

## Fancy Indexing ile Değer Değiştirme

Erişimin yanı sıra değiştirme de mümkün:


In [ ]:
# fancy_assign.py
x = np.arange(10)
i = np.array([2, 1, 8, 4])
x[i] = 99
print(x)



Herhangi bir atama operatörü:


In [ ]:
# fancy_assign_op.py
x[i] -= 10
print(x)



Tekrarlı indeksler beklenmedik sonuç verebilir:


In [ ]:
# fancy_repeat1.py
x = np.zeros(10)
x[[0, 0]] = [4, 6]
print(x)  # x[0] = 6 (4 kayboldu)



In [ ]:
# fancy_repeat2.py
i = [2, 3, 3, 4, 4, 4]
x[i] += 1
x  # x[3]=1, x[4]=1 — 2 ve 3 değil!



x[i] += 1 kısaltması x[i] = x[i] + 1. x[i] + 1 bir kez hesaplanır, sonra atama yapılır — artırma değil atama tekrarlanır.

Artırmanın her tekrarda uygulanmasını istiyorsanız ufunc'un at metodu:


In [ ]:
# add_at.py
x = np.zeros(10)
np.add.at(x, i, 1)
print(x)  # x[3]=2, x[4]=3



at belirtilen indekslerde operatörü yerinde uygular. Benzer: ufunc'ların reduceat metodu — NumPy ufunc dokümantasyonu.

## Örnek: Veriyi Kutulara Ayırma (Binning)

Fancy indexing fikirleriyle özel kutulu hesaplamalar yapılabilir. 100 değerin hangi kutuya düştüğünü bulmak:


In [ ]:
# binning_manual.py
rng = np.random.default_rng(seed=1701)
x = rng.normal(size=100)

bins = np.linspace(-5, 5, 20)
counts = np.zeros_like(bins)

# Her x için uygun kutuyu bul
i = np.searchsorted(bins, x)

# Her kutuya 1 ekle
np.add.at(counts, i, 1)
print("counts:", counts)



counts her kutudaki nokta sayısını verir — yani histogram. Matplotlib plt.hist tek satırda aynısını yapar:


```python
# plt.hist(x, bins, histtype='step')
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Matplotlib np.histogram kullanır. Karşılaştırma (notebook'ta %timeit):


In [ ]:
# binning_timing.py
import time

def bench(fn, n=50):
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1000

# 100 nokta
t_np = bench(lambda: np.histogram(x, bins))
t_custom = bench(lambda: np.add.at(np.zeros_like(bins), np.searchsorted(bins, x), 1))
print(f"100 nokta — np.histogram: {t_np:.3f} ms, özel: {t_custom:.3f} ms")

# 1M nokta
x_big = rng.normal(size=1_000_000)
t_np_big = bench(lambda: np.histogram(x_big, bins), n=5)
t_custom_big = bench(lambda: np.add.at(np.zeros_like(bins), np.searchsorted(bins, x_big), 1), n=5)
print(f"1M nokta — np.histogram: {t_np_big:.3f} ms, özel: {t_custom_big:.3f} ms")



Küçük veride özel algoritma daha hızlı olabilir; büyük veride np.histogram daha esnek ve optimize edilmiştir. Algoritmik verimlilik basit değildir — bkz. 2.8 Big-O Notasyonu.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      100 rastgele sayıdan 20 tanesini np.random.choice ile tekrarsız seçin.
    
      import numpy as np
X = np.random.randn(100, 2)
idx = np.random.choice(X.shape[0], 20, replace=False)
print("Seçilen:", len(idx), "  İlk 5 indeks:", idx[:5])

> **Not**
>
